In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_22_try_0.pickle

In [ ]:
%%PandasProfile
### cell 23 ###

def grab_subset_of_data_35(original_df, question_of_interest):
    # Vectorized column filtering and renaming, then drop rows with no responses
    cols = original_df.columns[original_df.columns.str.contains(question_of_interest)]
    if cols.empty:
        return original_df.iloc[0:0, 0:0]
    subset = original_df.loc[:, cols]
    # Rename by splitting once at last '-' and stripping
    subset.columns = subset.columns.str.rsplit('-', n=1).str[-1].str.lstrip()
    return subset.dropna(how='all')


def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_35(
        question_of_interest, include_2017=False):
    years = ['2018', '2019', '2020', '2021', '2022']
    dfs = [
        responses_df_2018_cell10,
        responses_df_2019_cell10,
        responses_df_2020,
        responses_df_2021,
        responses_df_2022_cell10
    ]
    if include_2017:
        years.insert(0, '2017')
        dfs.insert(0, responses_df_2017)

    parts = []
    for year, df in zip(years, dfs):
        # select & rename just this year's question cols
        cols = df.columns[df.columns.str.contains(question_of_interest)]
        if not cols.any():
            continue
        temp = df.loc[:, cols]
        temp.columns = temp.columns.str.rsplit('-', n=1).str[-1].str.lstrip()
        # drop respondents with no selections
        temp = temp.dropna(how='all')
        if temp.empty:
            continue
        temp['year'] = year
        parts.append(temp)
    # concat once with a simple RangeIndex rather than a MultiIndex
    combined = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=['year'])
    # count non-NA per choice, reindex to include all years
    counts = (
        combined
          .groupby('year', sort=False)
          .count()
          .reindex(years, fill_value=0)
          .reset_index()
    )
    return combined, counts


def convert_df_of_counts_to_percentages_35(df, df_counts):
    # Compute total respondents per year without sorting by count
    totals = df.groupby('year', sort=False).size().reindex(df_counts['year'], fill_value=0)
    # Divide and scale
    return (
        df_counts
          .set_index('year')
          .div(totals, axis=0)
          .mul(100)
          .reset_index()
    )

# Main pipeline
question_of_interest_cell35 = 'What programming languages do you use on a regular basis?'
language_df_combined, language_df_combined_counts = (
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_35(
        question_of_interest_cell35
    )
)
language_df_combined_percentages = convert_df_of_counts_to_percentages_35(
    language_df_combined, language_df_combined_counts
)

# Select & reshape in one chain, avoid extra rename step
langs = ['year', 'Python', 'SQL', 'C++', 'C', 'R', 'Java', 'Javascript', 'Other']
df_cell35 = (
    language_df_combined_percentages
      .loc[:, langs]
      .melt(id_vars='year', value_vars=langs[1:], var_name='', value_name='value')
      .sort_values(['year', 'value'])
)

df_cell35.info()